In [1]:
from datasets import load_dataset


c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [2]:
dataset = load_dataset("gretelai/synthetic_text_to_sql")

In [3]:
print(dataset)

DatasetDict({
    train: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 100000
    })
    test: Dataset({
        features: ['id', 'domain', 'domain_description', 'sql_complexity', 'sql_complexity_description', 'sql_task_type', 'sql_task_type_description', 'sql_prompt', 'sql_context', 'sql', 'sql_explanation'],
        num_rows: 5851
    })
})


In [4]:
sample = dataset["train"][0]

for key, value in sample.items():
    print("=" * 50)
    print("COLUMN:", key)
    print("TYPE:", type(value))
    
    if isinstance(value, str):
        print(value[:1000])
    else:
        print(value)

COLUMN: id
TYPE: <class 'int'>
5097
COLUMN: domain
TYPE: <class 'str'>
forestry
COLUMN: domain_description
TYPE: <class 'str'>
Comprehensive data on sustainable forest management, timber production, wildlife habitat, and carbon sequestration in forestry.
COLUMN: sql_complexity
TYPE: <class 'str'>
single join
COLUMN: sql_complexity_description
TYPE: <class 'str'>
only one join (specify inner, outer, cross)
COLUMN: sql_task_type
TYPE: <class 'str'>
analytics and reporting
COLUMN: sql_task_type_description
TYPE: <class 'str'>
generating reports, dashboards, and analytical insights
COLUMN: sql_prompt
TYPE: <class 'str'>
What is the total volume of timber sold by each salesperson, sorted by salesperson?
COLUMN: sql_context
TYPE: <class 'str'>
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volu

In [5]:
def format_example(example):
    prompt = f"""### Instruction
Generate a valid SQL query for the given database schema and question.

### Database
{example["sql_context"]}

### Question
{example["sql_prompt"]}

### SQL
{example["sql"]}"""

    return {
        "text": prompt
    }

In [6]:
formatted_dataset = dataset.map(format_example)

Map: 100%|██████████| 5851/5851 [00:00<00:00, 13058.71 examples/s]


In [7]:
print(formatted_dataset["train"][0]["text"])

### Instruction
Generate a valid SQL query for the given database schema and question.

### Database
CREATE TABLE salesperson (salesperson_id INT, name TEXT, region TEXT); INSERT INTO salesperson (salesperson_id, name, region) VALUES (1, 'John Doe', 'North'), (2, 'Jane Smith', 'South'); CREATE TABLE timber_sales (sales_id INT, salesperson_id INT, volume REAL, sale_date DATE); INSERT INTO timber_sales (sales_id, salesperson_id, volume, sale_date) VALUES (1, 1, 120, '2021-01-01'), (2, 1, 150, '2021-02-01'), (3, 2, 180, '2021-01-01');

### Question
What is the total volume of timber sold by each salesperson, sorted by salesperson?

### SQL
SELECT salesperson_id, name, SUM(volume) as total_volume FROM timber_sales JOIN salesperson ON timber_sales.salesperson_id = salesperson.salesperson_id GROUP BY salesperson_id, name ORDER BY total_volume DESC;


In [8]:
formatted_dataset = formatted_dataset.remove_columns(
    [col for col in formatted_dataset["train"].column_names if col != "text"]
)

In [9]:
print(formatted_dataset)

DatasetDict({
    train: Dataset({
        features: ['text'],
        num_rows: 100000
    })
    test: Dataset({
        features: ['text'],
        num_rows: 5851
    })
})


In [10]:
split_dataset = formatted_dataset["train"].train_test_split(
    test_size=0.1,
    seed=42
)

train_dataset = split_dataset["train"]
val_dataset = split_dataset["test"]

test_dataset = formatted_dataset["test"]

In [11]:
print(len(train_dataset))
print(len(val_dataset))
print(len(test_dataset))

90000
10000
5851


In [12]:
train_dataset = train_dataset.select(range(5000))
val_dataset = val_dataset.select(range(500))
test_dataset = test_dataset.select(range(500))

In [13]:
from transformers import AutoTokenizer

model_name = "Qwen/Qwen2.5-1.5B-Instruct"

tokenizer = AutoTokenizer.from_pretrained(model_name)

c:\Users\user\AppData\Local\Programs\Python\Python312\Lib\site-packages\huggingface_hub\file_download.py:144: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\user\.cache\huggingface\hub\models--Qwen--Qwen2.5-1.5B-Instruct. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


In [14]:
def count_tokens(example):
    tokens = tokenizer(example["text"])
    
    return {
        "num_tokens": len(tokens["input_ids"])
    }

train_dataset = train_dataset.map(count_tokens)
val_dataset = val_dataset.map(count_tokens)

Map: 100%|██████████| 500/500 [00:00<00:00, 2692.34 examples/s]


In [15]:
import pandas as pd

df = pd.DataFrame(train_dataset)

df["num_tokens"].describe()

count    5000.000000
mean      166.445400
std        61.696093
min        48.000000
25%       123.000000
50%       157.000000
75%       199.000000
max       600.000000
Name: num_tokens, dtype: float64